# RERANKING RAG Pipeline
### PDF → Chunking → FAISS → Broad Recall → Cross-Encoder Reranking → Answer

In [ ]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf sentence-transformers -q

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama
from sentence_transformers import CrossEncoder

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load PDF

In [2]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Sample Content:\n{pages[0].page_content[:500]}")

Total Pages: 30
Sample Content:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director Ge


## Step 2: Split into Chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Chunks: {len(chunks)}")
print(f"Sample Chunk:\n{chunks[0].page_content}")

Total Chunks: 136
Sample Chunk:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director General of Defence Research and
Development Organisation
In office
1992–1999
A. P. J. Abdul Kalam
Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl
kəˈlɑːm/ ⓘ UB-duul k ə -LAHM; 15 October 1931 –
27 July 2015) was an Indian aerospace scientist and
statesman who served as the president of India from
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a


## Step 3: Create Embeddings & Store in FAISS

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


## Step 4: Stage 1 — Broad Recall (Top 20 Candidates)

In [5]:
retriever = vector_store.as_retriever(search_kwargs={"k": 20})

query = "What role did Abdul Kalam play in India's missile program?"

candidates = retriever.invoke(query)

print(f"Broad Recall: {len(candidates)} candidates retrieved")
for i, doc in enumerate(candidates[:5]):
    print(f"\n--- Candidate {i+1} (Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content[:200])

Broad Recall: 20 candidates retrieved

--- Candidate 1 (Page 1) ---
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a
scientist and science administrator, ma

--- Candidate 2 (Page 4) ---
US$780 million in 2023) for the project titled Integrated Guided Missile Development Programme
(IGMDP) and appointed Kalam as its chief executive.[28] Kalam played a major role in the development
of m

--- Candidate 3 (Page 15) ---
realised-Nag-missile-dream-to-become-reality-next-year/articleshow/48267342.cms) from
the original on 3 January 2017. Retrieved 30 July 2015.
33. Sen, Amartya (2003). "India and the Bomb" (https://boo

--- Candidate 4 (Page 4) ---
Kalam greeting then prime minister
Vajpayee on 25 December 2003
major role in convincing the cabinet to conceal the true nature of these classified projects. His research
and leadership brought him re

--- Candidate 5 (Page 3) ---
Despite

## Step 5: Stage 2 — Cross-Encoder Reranking (Top 3)

In [6]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Score each candidate against the query
pairs = [[query, doc.page_content] for doc in candidates]
scores = cross_encoder.predict(pairs)

# Attach scores and sort by relevance
scored_docs = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)

# Pick top 3 reranked results
top_k = 3
reranked_docs = [doc for _, doc in scored_docs[:top_k]]

print(f"Reranked Top {top_k} Documents:")
for i, (score, doc) in enumerate(scored_docs[:top_k]):
    print(f"\n--- Reranked {i+1} | Score: {score:.4f} (Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content[:200])

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shiva\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1245.87it/s, Materializing 

Reranked Top 3 Documents:

--- Reranked 1 | Score: 8.3008 (Page 4) ---
US$780 million in 2023) for the project titled Integrated Guided Missile Development Programme
(IGMDP) and appointed Kalam as its chief executive.[28] Kalam played a major role in the development
of m

--- Reranked 2 | Score: 7.2302 (Page 1) ---
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a
scientist and science administrator, ma

--- Reranked 3 | Score: 6.7866 (Page 4) ---
Kalam greeting then prime minister
Vajpayee on 25 December 2003
major role in convincing the cabinet to conceal the true nature of these classified projects. His research
and leadership brought him re


## Step 6: Setup LLM & Query with Reranked Context

In [7]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

# Build context from reranked chunks
context = "\n\n".join([doc.page_content for doc in reranked_docs])

# Create prompt
prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

# Get response
response = llm.invoke(prompt)
print("Answer:", response.content)

Answer: Abdul Kalam played a major role in the development of India's missiles, including Agni and Prithvi, and was known as the "Missile Man of India" for his work on ballistic missile and launch vehicle technology.


## Reranked Source Chunks

In [8]:
for i, doc in enumerate(reranked_docs):
    print(f"\n--- Chunk {i+1} (Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content[:300])


--- Chunk 1 (Page 4) ---
US$780 million in 2023) for the project titled Integrated Guided Missile Development Programme
(IGMDP) and appointed Kalam as its chief executive.[28] Kalam played a major role in the development
of missiles including Agni, an intermediate range ballistic missile and Prithvi, the tactical surface-to

--- Chunk 2 (Page 1) ---
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a
scientist and science administrator, mainly at the
Defence Research and Development Organisation
(DRDO) and Indian Space Research Organisat

--- Chunk 3 (Page 4) ---
Kalam greeting then prime minister
Vajpayee on 25 December 2003
major role in convincing the cabinet to conceal the true nature of these classified projects. His research
and leadership brought him recognition in the 1980s, which prompted the government to initiate an
advanced missile programme unde


Test Questions

1. What year was Abdul Kalam born?
2. What was the name of the coronary stent Kalam co-developed?
3. Which institution did Kalam study aerospace engineering at?
4. How many electoral votes did Kalam receive in the 2002 presidential election?
5. What was Kalam's role at DRDO from 1992 to 1999?